<a href="https://colab.research.google.com/github/Deepr0gth/Flyrank_repo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaibachaopls-web/Flyrani_repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis: One row = one unique content item (content_id).
### Time Window: A fixed, trailing 90-day period.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
total_rows = len(df)
unique_content = df['content_id'].nunique()
is_perfect_grain = df['content_id'].is_unique

print(f"Total rows: {total_rows:,}")
print(f"Unique content IDs: {unique_content:,}")
print(f"Contract met (1 row = 1 content item): {is_perfect_grain}")


Total rows: 30,000
Unique content IDs: 30,000
Contract met (1 row = 1 content item): True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Sorting & Data Contract:

Features (Inputs): content_age_days, ctr, impressions_90d, position_tier, word_count

### Label (Target): is_decaying (Our custom binary variable).

### Context (Metadata): content_id (for row identification), client_id

### Excluded (Leakage): trend_pct, trend_direction, is_declining_label.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

df['is_decaying'] = np.where(df['trend_pct'] < -20, 1, 0)

features = ['content_age_days', 'ctr', 'impressions_90d', 'position_tier', 'word_count']
label = ['is_decaying']
context = ['content_id', 'client_id']
excluded_leakage = ['trend_pct', 'trend_direction', 'is_declining_label']

print(f"Are all selected Features present? {all(f in df.columns for f in features)}")
print(f"Is the Label present? {all(l in df.columns for l in label)}")
print(f"Are Context fields present? {all(c in df.columns for c in context)}")
print(f"Are Excluded fields actually in the raw data? {all(e in df.columns for e in excluded_leakage)}\n")

model_df = df[context + features + label]
display(model_df.head(3))


Are all selected Features present? True
Is the Label present? True
Are Context fields present? True
Are Excluded fields actually in the raw data? False



,content_id,client_id,content_age_days,ctr,impressions_90d,position_tier,word_count,is_decaying
0,content_304f48230142,client_f369cb89fc,187,0.76,3803,striking,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,445,0.05,15320,page_3_5,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,141,0.09,12581,page_3_5,3515.0,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Data Contract Verifications:
To prove the data contract outlined above is mathematically sound, the following queries verify our core assumptions against the actual dataset:

1. Grain Verification: Confirming content_id is a perfectly unique primary key (no duplicates).

2. Row Counts & Target Distribution: Verifying the total volume of data and ensuring our target label (is_decaying) has no missing values and a viable class balance.

3. Missing Values Check: Scanning our chosen features to identify any data gaps (specifically word_count) that we must formally address in the data pipeline before modeling.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ['content_age_days', 'ctr', 'impressions_90d', 'position_tier', 'word_count']
context = ['content_id', 'client_id']
label = ['is_decaying']
all_contract_cols = context + features + label

print(f"Total Rows: {len(df):,}")
print(f"Is 'content_id' perfectly unique? {df['content_id'].is_unique}")

print('\n')

missing_data = df[all_contract_cols].isnull().sum()
missing_data = missing_data[missing_data > 0]

if missing_data.empty:
    print("No missing values.")
else:
    print('Missing data found')
    print(missing_data)


Total Rows: 30,000
Is 'content_id' perfectly unique? True


Missing data found
word_count    7699
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### What this data can never tell us:

1. Unbalanced History: The dataset does not represent all clients equally. If one massive enterprise client makes up 40% of the rows, the model might just learn the specific decay patterns of that one website's architecture, rather than universal SEO truths.

2. GSC-Only Blindness: The features rely entirely on Google Search Console (organic search). It can never tell us if a page's total value dropped because it lost social media traffic, referral links, or paid ad support. It is strictly an organic search lens.

3. Window Overlaps & Granularity: Because the data provides pre-aggregated 90-day trailing metrics, we lack the daily time-series granularity to know exactly when within that 90-day window the decay started (e.g., a slow bleed over 3 months vs. a massive crash yesterday look the same in an aggregate sum).

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

client_distribution = df['client_id'].value_counts(normalize=True) * 100
top_client_pct = client_distribution.iloc[0]
bottom_client_pct = client_distribution.iloc[-1]

print(f"1. Client Imbalance: The top client provides {top_client_pct:.1f}% of the data, while the smallest provides {bottom_client_pct:.3f}%.")

all_cols = df.columns.tolist()
social_paid_cols = [col for col in all_cols if 'social' in col.lower() or 'paid' in col.lower()]

print(f"2. External Traffic Blindspot: Found {len(social_paid_cols)} columns tracking social or paid traffic.")

print(f"3. Time Granularity: Do we have day-by-day impression arrays? {'impressions_daily' in all_cols}")




1. Client Imbalance: The top client provides 23.4% of the data, while the smallest provides 0.010%.
2. External Traffic Blindspot: Found 0 columns tracking social or paid traffic.
3. Time Granularity: Do we have day-by-day impression arrays? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.